In [1]:
%load_ext autoreload
%autoreload 2

import glob
import numpy as np
import os
import json

In [9]:
# perform ensembling
def loadWeightsAndEnsemble(fileList, weight_name="Reweight_Step2.reweight.npy"):
    
    # load weights
    w = {}
    for file_path in fileList:

        # get job configuration
        with open(file_path) as f:
            conf = json.load(f)

        # load job type
        job_type = conf["new_mc_name"]
        w.setdefault(job_type, [])

        # append weight path
        weight_path = os.path.dirname(file_path)
        weight_file = os.path.join(weight_path, weight_name)
        weight = np.load(weight_file)
        w[job_type].append(weight)

    # ensemble
    e = {}
    for key, val in w.items():
        
        # stack
        f = np.stack(val, 1)

        # ensemble
        f = f/(1+f) # go back to raw NN predictions from w = f(x)/(1-x) -> f(x) = w/(1+w)
        f = np.median(f, axis=1) # take median weight
        f = f/(1-f) # go back to weights

        # store
        e[key] = f

    return e, w
    

# ensemble collection of theory reweightings
pathList = [
    "/pscratch/sd/b/badea/aleph/unfold-ee-logtau/ReweightMC/results/training-bf3b5fc3", 
    "/pscratch/sd/b/badea/aleph/unfold-ee-logtau/ReweightMC/results/training-b7199c7b"
]
fileList = []
for p in pathList:
    filePath = os.path.join(p, "*/*/conf.json")
    fileList.extend(sorted(glob.glob(filePath)))
print("Number of files:", len(fileList))

e1, w = loadWeightsAndEnsemble(fileList, weight_name = "Reweight_Step1.reweight.npy")
e2, w = loadWeightsAndEnsemble(fileList, weight_name = "Reweight_Step2.reweight.npy")

Number of files: 45


In [13]:
# # check health of ensembling
# for key, val in e.items():
#     print(key, np.mean(val), np.std(val), np.isnan(val).any(), val.shape)

# make folder inside the first pathList called ensemble, create a txt file with the ensemble paths in it, and save the ensemble weights in npy files
output_path = "./TEMP" # os.path.join(pathList[0], "ensemble")
os.makedirs(output_path, exist_ok=True)

# put ensemble paths into a txt file
output_file = os.path.join(output_path, "ensemble_paths.txt")
with open(output_file, "w") as f:
    for file_path in pathList:
        f.write(file_path + "\n")

# save ensemble weights
for step, e in enumerate([e1, e2]):
    for key, val in e.items():
        print(key, np.mean(val), np.std(val), np.isnan(val).any(), val.shape)
        output_file = os.path.join(output_path, f"Reweight_Step{step+1}_Ensemble_{key}.npy")
        np.save(output_file, val)
        print(f"Saved ensemble weights for {key} to {output_file} of size {val.shape}")
        print()

# for key, val in e2.items():
#     output_file = os.path.join(output_path, f"Reweight_Step2_Ensemble_{key}.npy")
#     np.save(output_file, val)
#     print(f"Saved ensemble weights for {key} to {output_file} of size {val.shape}")

Herwig 1.0044346 0.002625466 False (973769,)
Saved ensemble weights for Herwig to ./TEMP/Reweight_Step1_Ensemble_Herwig.npy of size (973769,)

Sherpa 1.0033139 0.0019263582 False (973769,)
Saved ensemble weights for Sherpa to ./TEMP/Reweight_Step1_Ensemble_Sherpa.npy of size (973769,)

Pythia8 1.0070825 0.0010229265 False (973769,)
Saved ensemble weights for Pythia8 to ./TEMP/Reweight_Step1_Ensemble_Pythia8.npy of size (973769,)

Herwig 0.85157835 1.7134631 False (973769,)
Saved ensemble weights for Herwig to ./TEMP/Reweight_Step2_Ensemble_Herwig.npy of size (973769,)

Sherpa 0.95192146 2.1999605 False (973769,)
Saved ensemble weights for Sherpa to ./TEMP/Reweight_Step2_Ensemble_Sherpa.npy of size (973769,)

Pythia8 0.9671248 1.2145764 False (973769,)
Saved ensemble weights for Pythia8 to ./TEMP/Reweight_Step2_Ensemble_Pythia8.npy of size (973769,)

